### Step 1: Prepare the input files

In [0]:
input_path = "/Volumes/workspace/default/my_files/chapter26_orders_input"
checkpoint_path = "/Volumes/workspace/default/my_files/chapter26_orders_checkpointer"

dbutils.fs.mkdirs(input_path)

True

### Step 2: Define the schema

In [0]:
from pyspark.sql.types import (
    StructField,StructType,IntegerType,StringType,DoubleType
)

schema = StructType([
    StructField("order_id",IntegerType(),True),
    StructField("customer_id",IntegerType(),True),
    StructField("order_amount",DoubleType(),True),
    StructField("order_status",StringType(),True)
])

### Step 3: Read the input files from the streaming folder

In [0]:
orders_stream_df = (
    spark.readStream
    .format("json")
    .schema(schema=schema)
    .option("maxFilesPerTrigger",2)
    .load(input_path)
)

### Step 4: Start the streaming query

- If we use processingTime="10 seconds"
databricks will thrown an error 
- [INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED] Trigger type ProcessingTime is not supported for this cluster type.
Use a different trigger type e.g. AvailableNow, Once. SQLSTATE: 0A000
- The difference between processingTime and availableNow is that processingTime will process the data in batches of 10 seconds, while availableNow will process the data as soon as it is available.

In [0]:
query = (
    orders_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation",checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.default.chapter26_orders")
)
query.awaitTermination()

In [0]:
%sql
select count(*) from workspace.default.chapter26_orders;

count(*)
60


### Step 5: Upload the remaining four files and run the query beacuse we are using availableNow so query will stop once after reading all the data we are unable to use processingTime beacuse of cluster constraints if we use processingTime then stream will run continuosly and read the for every 10 seconds if we define as processingTime = "10 seconds"

In [0]:
query = (
    orders_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation",checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.default.chapter26_orders")
)
query.awaitTermination()

In [0]:
%sql
select count(*) from workspace.default.chapter26_orders;

count(*)
100
